# Fine-Tuning TinyLLama with Generated MCQs

## Validate & Format Training Data

In [8]:
import json

with open("../data/generated_mcqs.jsonl") as f:
    raw_data = [json.loads(line) for line in f]

final_mcqs_full = []
for row in raw_data:
    # Validate
    invalid_choices = "A. A skin condition\nB. A viral infection\nC. A lung disease that causes airway inflammation\nD. A type of heart failure"
    if row["mcq_choices"] == invalid_choices and row["mcq_answer"] == "C":
        continue
    # Format
    prompt = row["question"].strip() + "\n" + row["mcq_choices"].strip()
    completion = row["mcq_reason"].strip() + "\n<answer>" + row["mcq_answer"].strip() + "</answer>"
    final_mcqs_full.append({"prompt": prompt, "completion": completion})

# Save validated & formatted dataset
with open("../data/final_mcqs_full.jsonl", "w") as f:
    for item in final_mcqs_full:
        f.write(json.dumps(item) + "\n")

## Split Data into Train/Test

In [9]:
import json
import random

# Load data
with open("../data/final_mcqs_full.jsonl", "r") as f:
    records = [json.loads(line) for line in f]

# Shuffle
random.seed(42)
random.shuffle(records)

# Split: 80% train, 20% test
split_idx = int(0.8 * len(records))
train_set = records[:split_idx]
test_set = records[split_idx:]

# Save to files
with open("../data/final_mcqs_train.jsonl", "w") as f:
    for entry in train_set:
        f.write(json.dumps(entry) + "\n")

with open("../data/final_mcqs_test.jsonl", "w") as f:
    for entry in test_set:
        f.write(json.dumps(entry) + "\n")

print(f"Split complete: {len(train_set)} train, {len(test_set)} test")


Split complete: 492 train, 123 test


## Supervised Fine-Tuning

In [ ]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# # Upload JSONL file
# uploaded = files.upload()
# file_path = list(uploaded.keys())[0]

# Get input from Google Drive
file_path = Path("/content/drive/MyDrive/final_mcqs_train.jsonl")

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Load train dataset (has 'prompt' and 'completion')
# dataset = load_dataset("json", data_files={"train": "../data/final_mcqs_train.jsonl"})["train"]
dataset = load_dataset("json", data_files={"train": file_path})["train"]

# Load tokenizer and model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Required for correct padding

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # load_in_8bit=True,
    # torch_dtype=torch.float16,
).to(device)

# Apply LoRA with PEFT
rank = 16
lora_alpha = rank * 4
peft_config = LoraConfig(target_modules="all-linear", bias="none", task_type="CAUSAL_LM", 
                         r=rank, lora_alpha=lora_alpha, lora_dropout=0.1)
model = get_peft_model(model, peft_config).to(device)
model.enable_input_require_grads()

# Tokenization
def format_and_tokenize(example):
    full_text = f"{example['prompt']}\n{example['completion']}"
    return tokenizer(full_text, truncation=True, padding="max_length", max_length=512)

tokenized_ds = dataset.map(format_and_tokenize)

# Google Drive output paths
output_dir = Path("/content/drive/MyDrive/tinyllama-mcq-lora")
logging_dir = Path("/content/drive/MyDrive/logs")

# Training config
training_args = TrainingArguments(
    gradient_checkpointing=True,
    # output_dir="../tinyllama-mcq-lora",
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    weight_decay=0.01,
    # fp16=True,
    learning_rate=2e-4,
    # logging_dir="../logs",
    logging_dir=logging_dir,
    logging_steps=50,
    save_strategy="epoch"
)

# Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)


Using device: mps


/opt/anaconda3/envs/AIHC/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


  0%|          | 0/369 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/opt/anaconda3/envs/AIHC/lib/python3.11/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
